##### Copyright 2026 Google LLC.

In [2]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Getting started with information grounding for Gemini models

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Grounding.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

In this notebook you will learn how to use information grounding with [Gemini models](https://ai.google.dev/gemini-api/docs/models/).

Information grounding is the process of connecting these models to specific, verifiable information sources to enhance the accuracy, relevance, and factual correctness of their responses. While LLMs are trained on vast amounts of data, this knowledge can be general, outdated, or lack specific context for particular tasks or domains. Grounding helps to bridge this gap by providing the LLM with access to curated, up-to-date information.

Here you will experiment with:
- Grounding information using <a href="#search_grounding">Google Search grounding</a>
- Grounding real-world information using <a href="#maps_grounding">Google Maps grounding</a>
- Adding <a href="#yt_links">YouTube links</a> to gather context information to your prompt
- Using <a href="#url_context">URL context</a> to include website, pdf or image URLs as context to your prompt

> **Note:** This notebook uses the [Interactions API](https://ai.google.dev/gemini-api/docs/interactions), the latest way to interact with Gemini models. Looking for the `generateContent` version? Check the [archive branch](https://github.com/google-gemini/cookbook/blob/archive/generate-content-api/quickstarts/Grounding.ipynb).

## Set up the SDK and the client

### Install SDK

This guide uses the [`google-genai`](https://pypi.org/project/google-genai) Python SDK to connect to the Gemini models.

In [ ]:
%pip install -U -q 'google-genai>=2.0.0'  # 2.0 for Interactions API

### Set up your API key

To run the following cell, your API key must be stored in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for an example.

In [11]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

### Select model and initialize SDK client

Select the model you want to use in this guide, either by selecting one in the list or writing it down. Keep in mind that some models, like the 2.5 ones are thinking models and thus take slightly more time to respond (cf. [thinking notebook](./Get_started_thinking.ipynb) for more details and in particular learn how to switch the thinking off).

In [13]:
from google import genai
from google.genai import types

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-2.5-flash-preview", "gemini-3.1-flash-lite", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

<a name="search_grounding"></a>

## Use Google Search grounding

Google Search grounding is particularly useful for queries that require current information or external knowledge. Using Google Search, Gemini can access nearly real-time information and better responses.

To enable Google Search, simply add the `google_search` tool in the `interactions.create`'s that way:
```
    config={
      "tools": [
        {
          "google_search": {}
        }
      ]
    },
```
> The response from `interactions.create` contains a list of `steps`. For text responses, access the output via `interaction.steps[-1].content[0].text`.


In [16]:
from IPython.display import display, HTML, Markdown

interaction = client.interactions.create(
    model=MODEL_ID,
    input="What was the latest Indian Premier League match and who won?",
    tools=[{"type": "google_search"}],
)

# Print the text response
text_output = next((o for o in interaction.steps if o.type == "text"), None)
if text_output:
    display(Markdown(f"**Response**:\n {text_output.text}"))

You can see that running the same prompt without search grounding gives you outdated information:

In [18]:
from IPython.display import display, Markdown

interaction = client.interactions.create(
    model=MODEL_ID,
    input="What was the latest Indian Premier League match and who won?",
)

text_output = next((o for o in interaction.steps if o.type == "text"), None)
if text_output:
    display(Markdown(text_output.text))

For more examples, please refer to the [dedicated notebook ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](./Search_Grounding.ipynb).

<a name="maps_grounding"></a>

## Use Google Maps grounding

Google Maps grounding allows you to easily incorporate location-aware functionality into your applications. When a prompt has context related to Maps data, the Gemini model uses Google Maps to provide factually accurate and fresh answers that are relevant to the specified location or general area.

To enable grounding with Google Maps, add the `google_maps` tool in the  `tools` argument of `interactions.create`, and optionally provide a structured location in the `tool_config`.

```python
client.models.interactions.create(
    ...,
    config=types.GenerateContentConfig(
      # Enable the tool.
      tools=[types.Tool(google_maps=types.GoogleMaps())],
      # Provide structured location.
      tool_config=types.ToolConfig(retrieval_config=types.RetrievalConfig(
            lat_lng=types.LatLng(
                latitude=34.050481, longitude=-118.248526))),
    )
)
```

In [22]:
from IPython.display import display, Markdown

interaction = client.interactions.create(
    model=MODEL_ID,
    input="Do any cafes around here do a good flat white? I will walk up to 20 minutes away",
    tools=[{"type": "google_maps"}],
)

text_output = next((o for o in interaction.steps if o.type == "text"), None)
if text_output:
    display(Markdown(f"### Response\n {text_output.text}"))

All grounded outputs require sources to be displayed after the response text. This code snippet will display the sources.

In [24]:
# Note: Grounding metadata (sources/citations) is not yet available 
# in the Interactions API. The function below works with the generate_content API.
# See Not_yet_in_Interactions_API.ipynb for details.

# def generate_sources(response):
#     grounding = response.candidates[0].grounding_metadata
#     ...
print("Grounding source display not yet available in Interactions API")

Grounding source display not yet available in Interactions API


The response also includes data you can use to assemble in-line links. See the [Grounding with Google Search docs](https://ai.google.dev/gemini-api/docs/google-search#attributing_sources_with_inline_citations) for an example of this.

### Render the contextual Google Maps widget

If you are building a web-based application, you can add an interactive widget that includes a map view, the contextual location, the places Gemini considered in the query, and review snippets.

To load the widget, perform all of the following steps.
1. [Acquire a Google Maps API key](https://developers.google.com/maps/documentation/javascript/get-api-key), enabled for the Places API and the Maps JavaScript API,
1. Request the widget token in your request (with `GoogleMaps(enable_widget=True)`),
1. [Load the Maps JavaScript API](https://developers.google.com/maps/documentation/javascript/load-maps-js-api) and enable the Places library,
1. Render the [`<gmp-place-contextual/>`](https://developers.google.com/maps/documentation/javascript/reference/places-widget#PlaceContextualElement) element, setting `context-token` to the value of the `google_maps_widget_context_token` returned in the Gemini API response.

Note that generating a widget can add additional latency to the response, so it is recommended that you do not enable the widget if you are not displaying it.

Assuming you have a Google Maps API key with both APIs enabled, the following code shows one way to render the widget.

In [27]:
from IPython.display import display, HTML

# Load or set your Maps API key here.
MAPS_API_KEY = userdata.get("MAPS_API_KEY")

# Google Maps widget rendering requires the Content-based API
# as the Interactions API does not yet return Maps widget data.
# This is for display purposes only.
interaction = client.interactions.create(
    model=MODEL_ID,
    input="Do any cafes around here do a good flat white? I will walk up to 20 minutes away",
    tools=[{"type": "google_maps"}],
)

text_output = next((o for o in interaction.steps if o.type == "text"), None)
if text_output:
    display(Markdown(text_output.text))

Running and rendering the above code will require a Maps API key. Once you have it working, the widget will look like this.

![Rendered contextual Places widget](https://storage.googleapis.com/generativeai-downloads/images/maps-widget.png)

<a name="yt_links"></a>

## Grounding with YouTube links

You can directly include a public YouTube URL in your prompt. The Gemini models will then process the video content to perform tasks like summarization and answering questions about the content.

This capability leverages Gemini's multimodal understanding, allowing it to analyze and interpret video data alongside any text prompts provided.

You do need to explicitly declare the video URL you want the model to process as part of the contents of the request using a `FileData` part. Here a simple interaction where you ask the model to summarize a YouTube video:

In [31]:
yt_link = "https://www.youtube.com/watch?v=XV1kOFo1C8M"

interaction = client.interactions.create(
    model=MODEL_ID,
    input="Summarize this video: " + yt_link,
    tools=[{"type": "url_context"}],
)

Markdown(interaction.steps[-1].content[0].text)


But you can also use the link as the source of truth for your request. In this example, you will first ask how Gemma models can help on chess games:

In [33]:
yt_link = "https://www.youtube.com/watch?v=XV1kOFo1C8M"

interaction = client.interactions.create(
    model=MODEL_ID,
    input="In 2 paragraphs, how can Gemma models help with chess games?: " + yt_link,
    tools=[{"type": "url_context"}],
)

Markdown(interaction.steps[-1].content[0].text)


Now your answer is more insightful for the topic you want, using the knowledge shared on the video and not necessarily available on the model knowledge.

<a name="url_context"></a>

## Grounding information using URL context

The URL Context tool empowers Gemini models to directly access and process content from specific web page URLs you provide within your API requests. This is incredibly interesting because it allows your applications to dynamically interact with live web information without needing you to manually pre-process and feed that content to the model.

URL Context is effective because it allows the models to base its responses and analysis directly on the content of the designated web pages. Instead of relying solely on its general training data or broad web searches (which are also valuable grounding tools), URL Context anchors the model's understanding to the specific information present at those URLs.

### Process website URLs

If you want Gemini to specifically ground its answers thanks to the content of a specific website, just add the urls in your prompt and enable the tool by adding it to your config:
```
config = {
  "tools": [
    {
      "url_context": {}
    }
  ],
}
```

You can add up to 20 links in your prompt.

In [38]:
prompt = """
  Based on https://ai.google.dev/gemini-api/docs/models, what are the key
  differences between Gemini 1.5, Gemini 2.0 and Gemini 2.5 models?
  Create a markdown table comparing the differences.
"""

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
    tools=[{"type": "url_context"}],
)

text_output = next((o for o in interaction.steps if o.type == "text"), None)
if text_output:
    display(Markdown(text_output.text))

You can see the status of the retrival using `url_context_metadata`:

In [40]:
# URL context metadata is available on the interaction object
# Note: The exact fields may differ from the generate_content API
print("URL context metadata is not yet available in the Interactions API.")
print("See Not_yet_in_Interactions_API.ipynb for current limitations.")

URL context metadata is not yet available in the Interactions API.
See Not_yet_in_Interactions_API.ipynb for current limitations.


### Add PDFs by URL

Gemini can also process PDFs from an URL. Here's an example:

In [42]:
prompt = """
  Can you give me an overview of the content of this pdf?
  https://abc.xyz/assets/cc/27/3ada14014efbadd7a58472f1f3f4/2025q2-alphabet-earnings-release.pdf

"""

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
    tools=[{"type": "url_context"}],
)

text_output = next((o for o in interaction.steps if o.type == "text"), None)
if text_output:
    display(Markdown(text_output.text.replace("$", r"\$")))

Grounding.ipynb:cell_41:15: SyntaxWarning: invalid escape sequence '\$'


### Add images by URL

Gemini can also process images from an URL. Here's an example:

In [44]:
prompt = """
  Can you help me name of the numbered parts of that instrument, in French?
  https://upload.wikimedia.org/wikipedia/commons/thumb/4/40/Trombone.svg/960px-Trombone.svg.png

"""

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
    tools=[{"type": "url_context"}],
)

text_output = next((o for o in interaction.steps if o.type == "text"), None)
if text_output:
    display(Markdown(text_output.text))

## Mix Search grounding and URL context

The different tools can also be use in conjunction by adding them both to the config. It's a good way to steer Gemini in the right direction and then let it do its magic using search grounding.

In [46]:
prompt = """
  Can you give me an overview of the content of this pdf?
  https://abc.xyz/assets/cc/27/3ada14014efbadd7a58472f1f3f4/2025q2-alphabet-earnings-release.pdf
  Search on the web for the reaction of the main financial analysts, what's the trend?
"""

interaction = client.interactions.create(
    model=MODEL_ID,
    input=prompt,
    tools=[{"type": "url_context"}, {"type": "google_search"}],
)

text_output = next((o for o in interaction.steps if o.type == "text"), None)
if text_output:
    display(Markdown(text_output.text.replace("$", r"\$")))

## Next steps

<a name="next_steps"></a>

* For more details about using Google Search grounding, check out the [Search Grounding cookbook](./Search_Grounding.ipynb).
* If you are looking for another scenarios using videos, take a look at the [Video understanding cookbook](./Video_understanding.ipynb).

Also check the other Gemini capabilities that you can find in the [Gemini quickstarts](https://github.com/google-gemini/cookbook/tree/main/quickstarts/).